# Model Explorer

Interactive tool for building intuition about how BE and SC model
parameters shape behaviour. Drag the sliders and watch the
psychometric curve, update matrix, and serial dependence profile
update in real time.

**This is a development tool, not a presentation notebook.**
For the narrative version, see `02_models_be_vs_sc.ipynb`.

### Panels
- **Left:** Psychometric curve (overall P(choose B) vs stimulus)
- **Centre:** Update matrix (how previous trial shifts next choice)
- **Right:** SD profile (column-mean of UM — the serial dependence signature)

### Controls
- **Model:** BE or SC
- **Distribution:** Uniform, Hard-A, Hard-B
- **Burn-in:** 0 or 1000 trials (lets belief converge before analysis)
- **Shared:** σ_percep, A_repulsion
- **BE-specific:** η_learning, η_relax
- **SC-specific:** γ, σ_update

In [ ]:
%matplotlib widget

import os, sys
from pathlib import Path

_NOTEBOOK_DIR = Path(os.path.abspath(''))
# Handle both notebooks/ and notebooks/dev/
if _NOTEBOOK_DIR.name == 'dev':
    _PROJECT_ROOT = _NOTEBOOK_DIR.parent.parent
else:
    _PROJECT_ROOT = _NOTEBOOK_DIR.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import (
    interact, interactive_output, HBox, VBox, Layout,
    FloatSlider, Dropdown, IntSlider, Output, ToggleButtons,
)
from IPython.display import display

from models.BE_core import BEParams, BEState, BEModel
from models.SC_core import SCParams, SCState, SCModel
from utils.stimulus_distributions import sample_distribution

from behav_utils.analysis.update_matrix import compute_update_matrix
from behav_utils.analysis.psychometry import fit_psychometric
from behav_utils.analysis.utils import cumulative_gaussian
from behav_utils.plotting.styles import apply_style, COLOURS, UM_CMAP

apply_style()
print('Ready.')

## Simulation engine

In [ ]:
N_TRIALS = 500
N_BINS = 8
SEED = 42

BE_COL = COLOURS.get('BE', '#2196F3')
SC_COL = COLOURS.get('SC', '#FF9800')


def _generate_stimuli(distribution, n_trials=N_TRIALS, seed=SEED):
    """Generate stimuli + categories for the given distribution."""
    rng = np.random.default_rng(seed)
    if distribution == 'Uniform':
        stimuli = rng.uniform(-1, 1, n_trials)
        categories = (stimuli > 0).astype(int)
    elif distribution == 'Hard-A':
        stimuli, categories = sample_distribution(n_trials, 'hard_a', rng=rng)
    elif distribution == 'Hard-B':
        stimuli, categories = sample_distribution(n_trials, 'hard_b', rng=rng)
    else:
        raise ValueError(f'Unknown distribution: {distribution}')
    return stimuli, categories


def simulate(
    model, distribution, burn_in,
    sigma_percep, A_repulsion,
    eta_learning, eta_relax,
    gamma, sigma_update,
):
    """
    Run one simulation and return all arrays needed for plotting.

    Returns dict with: stimuli, categories, choices, um, accuracy,
    psych_params, sd_profile, bin_centres, model_colour.
    """
    stimuli, categories = _generate_stimuli(distribution)
    rng = np.random.default_rng(SEED + 1)

    if model == 'BE':
        params = BEParams(
            sigma_percep=sigma_percep, A_repulsion=A_repulsion,
            eta_learning=eta_learning, eta_relax=eta_relax,
        )
        state = BEModel.create_initial_state(
            burn_in=burn_in, params=params, seed=SEED)
        choices, p_B, _, _ = BEModel.simulate_session(
            params, state, stimuli, categories, rng,
            return_history=False,
        )
        colour = BE_COL
    else:
        params = SCParams(
            sigma_percep=sigma_percep, A_repulsion=A_repulsion,
            gamma=gamma, sigma_update=sigma_update,
        )
        state = SCModel.create_initial_state(
            burn_in=burn_in, params=params, seed=SEED)
        choices, p_B, _, _ = SCModel.simulate_session(
            params, state, stimuli, categories, rng,
            return_history=False,
        )
        colour = SC_COL

    # Accuracy
    valid = ~np.isnan(choices)
    accuracy = np.mean(choices[valid] == categories[valid])

    # Update matrix
    um, _, info = compute_update_matrix(
        stimuli, choices, categories, n_bins=N_BINS,
        trial_filter='post_correct',
    )

    # Psychometric fit
    psych = fit_psychometric(stimuli[valid], choices[valid])

    # SD profile (mean across columns = mean over current-stimulus bins)
    sd_profile = np.nanmean(um, axis=0)

    # Bin centres for SD profile x-axis
    edges = np.linspace(-1, 1, N_BINS + 1)
    bin_centres = (edges[:-1] + edges[1:]) / 2

    return {
        'stimuli': stimuli, 'categories': categories,
        'choices': choices, 'p_B': p_B,
        'um': um, 'accuracy': accuracy,
        'psych': psych, 'sd_profile': sd_profile,
        'bin_centres': bin_centres, 'colour': colour,
        'model': model, 'distribution': distribution,
    }

## Single model explorer

In [ ]:
# ── Widgets ──────────────────────────────────────────────────────────────────

slider_layout = Layout(width='340px')
style = {'description_width': '120px'}

w_model = ToggleButtons(
    options=['BE', 'SC'], value='BE',
    description='Model:', style=style,
)
w_dist = Dropdown(
    options=['Uniform', 'Hard-A', 'Hard-B'], value='Uniform',
    description='Distribution:', style=style, layout=slider_layout,
)
w_burn_in = IntSlider(
    value=1000, min=0, max=2000, step=100,
    description='Burn-in:', style=style, layout=slider_layout,
)

# Shared perceptual
w_sigma = FloatSlider(
    value=0.15, min=0.01, max=0.60, step=0.01,
    description='σ_percep:', style=style, layout=slider_layout,
    readout_format='.2f',
)
w_repulsion = FloatSlider(
    value=0.10, min=0.00, max=0.50, step=0.01,
    description='A_repulsion:', style=style, layout=slider_layout,
    readout_format='.2f',
)

# BE-specific
w_eta_learn = FloatSlider(
    value=0.30, min=0.01, max=0.90, step=0.01,
    description='η_learning:', style=style, layout=slider_layout,
    readout_format='.2f',
)
w_eta_relax = FloatSlider(
    value=0.15, min=0.01, max=0.90, step=0.01,
    description='η_relax:', style=style, layout=slider_layout,
    readout_format='.2f',
)

# SC-specific
w_gamma = FloatSlider(
    value=0.50, min=0.01, max=1.00, step=0.01,
    description='γ:', style=style, layout=slider_layout,
    readout_format='.2f',
)
w_sigma_update = FloatSlider(
    value=0.20, min=0.01, max=0.60, step=0.01,
    description='σ_update:', style=style, layout=slider_layout,
    readout_format='.2f',
)


# ── Show/hide model-specific sliders ─────────────────────────────────────────

be_box = VBox([w_eta_learn, w_eta_relax])
sc_box = VBox([w_gamma, w_sigma_update])

def _toggle_sliders(change):
    be_box.layout.display = 'flex' if change['new'] == 'BE' else 'none'
    sc_box.layout.display = 'flex' if change['new'] == 'SC' else 'none'

w_model.observe(_toggle_sliders, names='value')
sc_box.layout.display = 'none'  # initial state


# ── Plot function ────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
fig.canvas.header_visible = False
fig.subplots_adjust(wspace=0.35)


def update(
    model, distribution, burn_in,
    sigma_percep, A_repulsion,
    eta_learning, eta_relax,
    gamma, sigma_update,
):
    result = simulate(
        model, distribution, burn_in,
        sigma_percep, A_repulsion,
        eta_learning, eta_relax,
        gamma, sigma_update,
    )

    col = result['colour']
    valid = ~np.isnan(result['choices'])
    stim_v = result['stimuli'][valid]
    ch_v = result['choices'][valid]

    for ax in axes:
        ax.clear()

    # ── Panel 1: Psychometric ────────────────────────────────────────────
    ax = axes[0]
    # Binned data
    n_psych_bins = 12
    edges = np.linspace(-1, 1, n_psych_bins + 1)
    centres = (edges[:-1] + edges[1:]) / 2
    p_b_binned = np.full(n_psych_bins, np.nan)
    for i in range(n_psych_bins):
        mask = (stim_v >= edges[i]) & (stim_v < edges[i + 1])
        if mask.sum() > 0:
            p_b_binned[i] = np.mean(ch_v[mask])
    ax.scatter(centres, p_b_binned, color=col, s=30, zorder=3, alpha=0.8)

    # Fitted curve
    psych = result['psych']
    if psych.get('success', True):
        x_fit = np.linspace(-1, 1, 200)
        y_fit = cumulative_gaussian(
            x_fit, psych['mu'], psych['sigma'],
            psych['lapse_low'], psych['lapse_high'],
        )
        ax.plot(x_fit, y_fit, color=col, lw=2)

    ax.axhline(0.5, ls=':', color='grey', lw=0.6)
    ax.axvline(0, ls=':', color='grey', lw=0.6)
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel('Stimulus')
    ax.set_ylabel('P(choose B)')
    pse_str = f'{psych["mu"]:.2f}' if psych.get('success', True) else '?'
    ax.set_title(
        f'Psychometric  (acc={result["accuracy"]:.2f}, PSE={pse_str})',
        fontsize=10,
    )

    # ── Panel 2: Update Matrix ───────────────────────────────────────────
    ax = axes[1]
    um = result['um']
    abs_max = max(np.nanmax(np.abs(um)), 0.01)
    im = ax.imshow(
        um, cmap=UM_CMAP, vmin=-abs_max, vmax=abs_max,
        origin='lower', aspect='equal',
    )
    tick_labels = [f'{c:.1f}' for c in result['bin_centres']]
    ax.set_xticks(range(N_BINS))
    ax.set_xticklabels(tick_labels, fontsize=6, rotation=45)
    ax.set_yticks(range(N_BINS))
    ax.set_yticklabels(tick_labels, fontsize=6)
    ax.set_xlabel('Previous stimulus')
    ax.set_ylabel('Current stimulus')
    ax.set_title('Update Matrix', fontsize=10)

    # ── Panel 3: SD Profile ──────────────────────────────────────────────
    ax = axes[2]
    sd = result['sd_profile']
    ax.plot(result['bin_centres'], sd, 'o-', color=col, markersize=5, lw=1.5)
    ax.axhline(0, ls='-', color='grey', lw=0.5)
    ax.set_xlabel('Previous stimulus bin')
    ax.set_ylabel('Mean ΔP(B)')
    y_lim = max(np.nanmax(np.abs(sd)) * 1.3, 0.03)
    ax.set_ylim(-y_lim, y_lim)
    ax.set_title('SD Profile', fontsize=10)

    fig.suptitle(
        f'{model} — {distribution}  (burn-in={burn_in})',
        fontsize=12, fontweight='bold', y=1.02,
    )
    fig.canvas.draw_idle()


# ── Wire up ──────────────────────────────────────────────────────────────────

out = interactive_output(update, {
    'model': w_model, 'distribution': w_dist, 'burn_in': w_burn_in,
    'sigma_percep': w_sigma, 'A_repulsion': w_repulsion,
    'eta_learning': w_eta_learn, 'eta_relax': w_eta_relax,
    'gamma': w_gamma, 'sigma_update': w_sigma_update,
})

controls = VBox([
    w_model, w_dist, w_burn_in,
    w_sigma, w_repulsion,
    be_box, sc_box,
])

display(HBox([controls, out]))

## Side-by-side comparison

Compare BE and SC with matched perceptual parameters on the same stimuli.
Useful for understanding what the UM signature differences actually look like.

In [ ]:
# ── Widgets ──────────────────────────────────────────────────────────────────

c_dist = Dropdown(
    options=['Uniform', 'Hard-A', 'Hard-B'], value='Uniform',
    description='Distribution:', style=style, layout=slider_layout,
)
c_burn = IntSlider(
    value=1000, min=0, max=2000, step=100,
    description='Burn-in:', style=style, layout=slider_layout,
)
c_sigma = FloatSlider(
    value=0.15, min=0.01, max=0.60, step=0.01,
    description='σ_percep:', style=style, layout=slider_layout,
    readout_format='.2f',
)
c_repulsion = FloatSlider(
    value=0.10, min=0.00, max=0.50, step=0.01,
    description='A_repulsion:', style=style, layout=slider_layout,
    readout_format='.2f',
)
c_eta_learn = FloatSlider(
    value=0.35, min=0.01, max=0.90, step=0.01,
    description='η_learning:', style=style, layout=slider_layout,
    readout_format='.2f',
)
c_eta_relax = FloatSlider(
    value=0.12, min=0.01, max=0.90, step=0.01,
    description='η_relax:', style=style, layout=slider_layout,
    readout_format='.2f',
)
c_gamma = FloatSlider(
    value=0.95, min=0.01, max=1.00, step=0.01,
    description='γ:', style=style, layout=slider_layout,
    readout_format='.2f',
)
c_sigma_update = FloatSlider(
    value=0.30, min=0.01, max=0.60, step=0.01,
    description='σ_update:', style=style, layout=slider_layout,
    readout_format='.2f',
)


# ── Plot ─────────────────────────────────────────────────────────────────────

fig2, axes2 = plt.subplots(2, 3, figsize=(14, 8))
fig2.canvas.header_visible = False
fig2.subplots_adjust(hspace=0.45, wspace=0.35)


def update_comparison(
    distribution, burn_in,
    sigma_percep, A_repulsion,
    eta_learning, eta_relax,
    gamma, sigma_update,
):
    be = simulate(
        'BE', distribution, burn_in,
        sigma_percep, A_repulsion,
        eta_learning, eta_relax,
        gamma, sigma_update,
    )
    sc = simulate(
        'SC', distribution, burn_in,
        sigma_percep, A_repulsion,
        eta_learning, eta_relax,
        gamma, sigma_update,
    )

    for row_idx, (result, label) in enumerate([(be, 'BE'), (sc, 'SC')]):
        col = result['colour']
        valid = ~np.isnan(result['choices'])
        stim_v = result['stimuli'][valid]
        ch_v = result['choices'][valid]

        for ax in axes2[row_idx]:
            ax.clear()

        # Psychometric
        ax = axes2[row_idx, 0]
        n_bins_p = 12
        edges = np.linspace(-1, 1, n_bins_p + 1)
        centres = (edges[:-1] + edges[1:]) / 2
        for i in range(n_bins_p):
            mask = (stim_v >= edges[i]) & (stim_v < edges[i + 1])
            if mask.sum() > 0:
                ax.scatter(centres[i], np.mean(ch_v[mask]),
                           color=col, s=25, zorder=3, alpha=0.8)

        psych = result['psych']
        if psych.get('success', True):
            x_fit = np.linspace(-1, 1, 200)
            y_fit = cumulative_gaussian(
                x_fit, psych['mu'], psych['sigma'],
                psych['lapse_low'], psych['lapse_high'],
            )
            ax.plot(x_fit, y_fit, color=col, lw=2)

        ax.axhline(0.5, ls=':', color='grey', lw=0.6)
        ax.axvline(0, ls=':', color='grey', lw=0.6)
        ax.set_xlim(-1.05, 1.05)
        ax.set_ylim(-0.02, 1.02)
        ax.set_xlabel('Stimulus')
        ax.set_ylabel('P(choose B)')
        ax.set_title(
            f'{label}  (acc={result["accuracy"]:.2f})',
            fontsize=10, color=col, fontweight='bold',
        )

        # UM
        ax = axes2[row_idx, 1]
        um = result['um']
        abs_max = max(np.nanmax(np.abs(um)), 0.01)
        ax.imshow(
            um, cmap=UM_CMAP, vmin=-abs_max, vmax=abs_max,
            origin='lower', aspect='equal',
        )
        tick_labels = [f'{c:.1f}' for c in result['bin_centres']]
        ax.set_xticks(range(N_BINS))
        ax.set_xticklabels(tick_labels, fontsize=6, rotation=45)
        ax.set_yticks(range(N_BINS))
        ax.set_yticklabels(tick_labels, fontsize=6)
        ax.set_xlabel('Previous stimulus')
        ax.set_ylabel('Current stimulus')
        ax.set_title('Update Matrix', fontsize=10)

        # SD profile
        ax = axes2[row_idx, 2]
        sd = result['sd_profile']
        ax.plot(result['bin_centres'], sd, 'o-', color=col, markersize=5, lw=1.5)
        ax.axhline(0, ls='-', color='grey', lw=0.5)
        ax.set_xlabel('Previous stimulus bin')
        ax.set_ylabel('Mean ΔP(B)')
        y_lim = max(np.nanmax(np.abs(sd)) * 1.3, 0.03)
        ax.set_ylim(-y_lim, y_lim)
        ax.set_title('SD Profile', fontsize=10)

    fig2.suptitle(
        f'BE vs SC — {distribution}  (burn-in={burn_in})',
        fontsize=12, fontweight='bold', y=1.01,
    )
    fig2.canvas.draw_idle()


out2 = interactive_output(update_comparison, {
    'distribution': c_dist, 'burn_in': c_burn,
    'sigma_percep': c_sigma, 'A_repulsion': c_repulsion,
    'eta_learning': c_eta_learn, 'eta_relax': c_eta_relax,
    'gamma': c_gamma, 'sigma_update': c_sigma_update,
})

controls2 = VBox([
    c_dist, c_burn,
    c_sigma, c_repulsion,
    c_eta_learn, c_eta_relax,
    c_gamma, c_sigma_update,
])

display(HBox([controls2, out2]))

## Quick reference: what to look for

**BE signature in the UM:**
Serial dependence is strongest near the boundary (centre rows/columns)
and weakest at extremes. The diagonal stripe reflects boundary-tracking:
a near-boundary stimulus on the previous trial shifts the boundary estimate,
biasing the next choice.

**SC signature in the UM:**
Serial dependence is strongest at the extremes and weakest near the boundary.
This reflects distributional learning: an extreme previous stimulus
adds mass to one category's distribution far from the boundary, shifting
the normalised posterior everywhere — including near-boundary trials.

**Key parameter effects:**
- `η_learning` (BE): higher = stronger UM contrast, more recency bias
- `η_relax` (BE): higher = boundary drifts back to centre faster
- `γ` (SC): higher = stronger distributional updating
- `σ_update` (SC): higher = broader KDE kernel, smoother category distributions
- `σ_percep` (both): higher = shallower psychometric slope
- `A_repulsion` (both): higher = stimulus repulsion from previous percept

**Distribution effects:**
- Hard-A/B shift the PSE toward the over-represented side
- SC predicts this normatively; BE produces it via recency
- The UM structure changes with distribution (more trials near boundary = better UM estimation there)